# Manually specified MNIST classifier

University of Michigan EECS 498/598, Winter 2022. Compact companion to the completed course assignment.

**Static validation only; not executed.** No ordinary-notebook outputs or scores are copied here. Run cells in order only when execution is intended; training variants retain the ordinary notebook parameters. Existing course dependencies and data are required; no installation or repository cloning is performed. Results are isolated under `core_results/challenge_problem`.

## Setup

In [ ]:
from pathlib import Path
import os
import sys
import importlib

candidates = [Path.cwd(), *Path.cwd().parents, Path('/content/drive/MyDrive/umich-dlcv-wi2022')]
PROJECT_ROOT = next((p for p in candidates if (p / 'assignments' / 'a2' / 'challenge_problem_core.ipynb').is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Open the canonical project or assignment directory; mount Drive first in Colab.')
ASSIGNMENT_DIR = PROJECT_ROOT / 'assignments' / 'a2'
if 'eecs598' in sys.modules and Path(sys.modules['eecs598'].__file__).resolve().parent != (ASSIGNMENT_DIR / 'eecs598').resolve():
    raise RuntimeError('Restart the kernel before switching assignments: eecs598 differs across assignments.')
os.chdir(ASSIGNMENT_DIR)
sys.path.insert(0, str(ASSIGNMENT_DIR))
sys.modules.setdefault('imp', importlib)
import torch
import matplotlib.pyplot as plt
import eecs598
from eecs598 import reset_seed

device = 'cuda' if torch.cuda.is_available() else 'cpu'
RESULTS_DIR = ASSIGNMENT_DIR / 'core_results' / 'challenge_problem'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
reset_seed(0)
print('Device:', device)


## Hand-specified weights (no gradient training)

In [ ]:
from eecs598.a2_challenge_problem import get_w1, get_w2, display_templates, display_w2, evaluate_MNIST
# Retained hand-specified geometric strokes for digits 0, 1, 2, 3, 4, 6, 7.
# The supplied image encodes those weights directly; it is not a GUI export.
path_to_weights = ASSIGNMENT_DIR / 'hand_drawn_weights.jpeg'
if not path_to_weights.is_file():
    raise FileNotFoundError(path_to_weights)
w1 = get_w1(str(path_to_weights))
w2 = get_w2(str(path_to_weights))
display_templates(w1)
display_w2(w2)


## Evaluate prepared MNIST test data

In [ ]:
mnist_raw = ASSIGNMENT_DIR / 'mnist_dataset' / 'MNIST' / 'raw'
required_mnist = ['t10k-images-idx3-ubyte', 't10k-labels-idx1-ubyte', 'train-images-idx3-ubyte', 'train-labels-idx1-ubyte']
if not all((mnist_raw / name).is_file() for name in required_mnist):
    raise FileNotFoundError('Prepare the MNIST raw cache under assignments/a2/mnist_dataset before evaluation.')
accuracy, correct_digits, incorrect_digits = evaluate_MNIST(w1, w2)
print('MNIST test accuracy (%):', 100 * accuracy)
torch.save({'accuracy': accuracy}, RESULTS_DIR / 'mnist_metrics.pt')


## Most confident correct / incorrect examples

In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(12, 4))
for row, (label, examples) in enumerate([('Correct', correct_digits), ('Incorrect', incorrect_digits)]):
    for col, ax in enumerate(axes[row]):
        if col < len(examples):
            ax.imshow(examples[col], cmap='gray')
        ax.set_title(label)
        ax.axis('off')
fig.savefig(RESULTS_DIR / 'mnist_examples.png', bbox_inches='tight')
plt.show()
